# Stage 1 -- Python Dataset Statistics

Produces `outputs/stage1_python_stats.xlsx` with sheets: **Py_Real**, **Py_Synth**, **Py_AI**, **Filters**, **Legend**.

**Prerequisites:** run `00_download_datasets.ipynb` first.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW      = ROOT / 'data' / 'raw'
CWE_XML  = ROOT / 'data' / 'cwec_latest.xml'
OUT_DIR  = ROOT / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
XLSX_PATH = OUT_DIR / 'stage1_python_stats.xlsx'

DATASET_FILTERS = {
    'CVEfixes(Python)':  {'branch': 'real',  'language': 'Python', 'source': 'HuggingFace hitoshura25/cvefixes', 'positives': 'language==Python, vulnerable_code non-empty, CWE non-empty, single-function commit', 'negatives': 'none', 'cwe_source': 'cwe_id column (NVD join)', 'filters_applied': 'language filter, NVD placeholder drop, single-function commit', 'safe_type': 'none', 'label_quality': '3/5', 'notes': ''},
    'PatchEval':         {'branch': 'real',  'language': 'Python', 'source': 'GitHub bytedance/PatchEval', 'positives': 'vul_func where programming_language==Python', 'negatives': 'fix_func (fix-paired safe)', 'cwe_source': 'cwe_info dict keys', 'filters_applied': 'language==Python filter, CWE non-empty', 'safe_type': 'fix-paired', 'label_quality': '4/5', 'notes': 'optional docker_verified_only flag'},
    'CrossVul(Python)':  {'branch': 'real',  'language': 'Python', 'source': 'Zenodo crossvul.zip', 'positives': 'bad_* files (vulnerable functions)', 'negatives': 'good_* files (fixed functions)', 'cwe_source': 'CWE-NNN directory name', 'filters_applied': 'py/ folder filter', 'safe_type': 'fix-paired', 'label_quality': '3/5', 'notes': ''},
    'PyVul':             {'branch': 'real',  'language': 'Python', 'source': 'GitHub billquan/PyVul', 'positives': 'code_before (Python), CWE from commits_cwe_map.json', 'negatives': 'code_after (fix-paired)', 'cwe_source': 'commits_cwe_map.json keyed by commit URL', 'filters_applied': 'programming_language==Python, CWE non-empty', 'safe_type': 'fix-paired', 'label_quality': '3/5', 'notes': 'function_level_dataset.out joined with commits_cwe_map.json'},
    'SVEN(Python)':      {'branch': 'real',  'language': 'Python', 'source': 'HuggingFace bstee615/sven', 'positives': 'func_src_before, CWE from vul_type', 'negatives': 'func_src_after (fix-paired)', 'cwe_source': 'vul_type column (cwe-NNN format)', 'filters_applied': 'language==Python (from file extension), empty vul_type dropped', 'safe_type': 'fix-paired', 'label_quality': '4/5', 'notes': ''},
    'OWASP(Python)':     {'branch': 'synth', 'language': 'Python', 'source': 'OWASP GitHub BenchmarkPython', 'positives': 'real vulnerability=true in expectedresults-0.1.csv', 'negatives': 'real vulnerability=false', 'cwe_source': 'cwe column (bare digit, normalised to CWE-NNN)', 'filters_applied': 'CSV join on filename (case-insensitive stem match)', 'safe_type': 'pure', 'label_quality': '4/5', 'notes': 'preliminary Python version; limited coverage'},
    'LLMSecEval':        {'branch': 'ai',   'language': 'Python', 'source': 'Zenodo 5225651 (vulnerable) + GitHub tuhh-softsec/LLMSecEval (safe)', 'positives': 'gen_scenario/*.py (non-reject Copilot completions, 12 CWEs)', 'negatives': '.py files in CWE-NNN/Secure/ dirs (18 CWEs)', 'cwe_source': 'cwe-NNN directory name', 'filters_applied': '.py extension, .reject excluded by glob', 'safe_type': 'pure', 'label_quality': '3/5', 'notes': 'Copilot-generated; ~40% actually vulnerable per paper; all labeled as vuln'},
    'SecurityEval':      {'branch': 'ai',   'language': 'Python', 'source': 'GitHub s2e-lab/SecurityEval', 'positives': 'all samples (dataset is vulnerable-only)', 'negatives': 'none', 'cwe_source': 'CWE field / directory name', 'filters_applied': 'code non-empty, CWE non-empty', 'safe_type': 'none', 'label_quality': '3/5', 'notes': 'no safe samples; LLM-generated insecure code'},
    'CAPEC_LLM(Python)': {'branch': 'ai',   'language': 'Python', 'source': 'GitHub llmForCapec/CAPECDatasetsLLM', 'positives': 'code_snippet detected as Python, CWE from description', 'negatives': 'none', 'cwe_source': 'CWE-NNN regex on description', 'filters_applied': 'language detection (Python keywords), CWE regex', 'safe_type': 'none', 'label_quality': '2/5', 'notes': 'LLM-generated; CWE from CAPEC description'},
}

print(f'Root: {ROOT}')
print(f'Output: {XLSX_PATH}')

## 1. CWE XML -- download if missing

In [2]:
import io, urllib.request, zipfile

CWE_ZIP_URL = 'https://cwe.mitre.org/data/xml/cwec_latest.xml.zip'

if CWE_XML.exists():
    print(f'CWE XML present ({CWE_XML.stat().st_size / 1e6:.1f} MB) -- skipping download.')
else:
    print('Downloading CWE XML from MITRE ...')
    with urllib.request.urlopen(CWE_ZIP_URL, timeout=60) as resp:
        data = resp.read()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        xml_name = next(n for n in zf.namelist() if n.endswith('.xml'))
        CWE_XML.write_bytes(zf.read(xml_name))
    print(f'Saved: {CWE_XML} ({CWE_XML.stat().st_size / 1e6:.1f} MB)')

CWE XML present (16.1 MB) -- skipping download.


## 2. Load CWE Navigator

In [3]:
from ingestion.cwe_navigator import CWENavigator

nav = CWENavigator(str(CWE_XML))
print(f'Loaded: {len(nav.weaknesses):,} weaknesses, {len(nav.categories):,} categories')

_parent_ids: set[str] = {p for parents in nav.child_of.values() for p in parents}

def _strip(cwe_str: str) -> str:
    return cwe_str.removeprefix('CWE-').removeprefix('cwe-').strip()

def cwe_label(cwe_str: str) -> str:
    num = _strip(cwe_str)
    if num in nav.categories or num in nav.views:
        return 'category'
    if num not in nav.weaknesses:
        return 'unknown'
    if nav.get_element_name(num).startswith('DEPRECATED:'):
        return 'deprecated'
    return 'leaf' if num not in _parent_ids else 'non-leaf'

Loaded: 969 weaknesses, 420 categories


## 3. Extract samples (skip if files missing)

In [ ]:
from collections import Counter
from ingestion.schema          import FunctionSample
from ingestion.cvefixes        import extract_cvefixes
from ingestion.crossvul        import extract_crossvul
from ingestion.sven            import extract_sven
from ingestion.patcheval       import extract_patcheval
from ingestion.pyvul           import extract_pyvul
from ingestion.owasp_benchmark import extract_owasp_benchmark
from ingestion.llmseceval      import extract_llmseceval
from ingestion.security_eval   import extract_security_eval
from ingestion.capec_llm       import extract_capec_llm

def _try(name, loader):
    try:
        s = loader()
        print(f'  {name:24s}: {len(s):>7,} samples')
        return s
    except FileNotFoundError as e:
        print(f'  {name:24s}: SKIPPED -- {e}')
        return []

collections: dict[str, list[FunctionSample]] = {}

collections['CVEfixes(Python)']   = _try('CVEfixes(Python)',   lambda: extract_cvefixes(RAW/'cvefixes', language='Python'))
collections['PatchEval']          = _try('PatchEval',          lambda: extract_patcheval(RAW/'patcheval'))
collections['CrossVul(Python)']   = _try('CrossVul(Python)',   lambda: extract_crossvul(RAW/'crossvul.zip', language='Python'))
collections['PyVul']              = _try('PyVul',              lambda: extract_pyvul(RAW/'pyvul'))
collections['SVEN(Python)']       = _try('SVEN(Python)',       lambda: extract_sven(RAW/'sven', language='Python'))
collections['OWASP(Python)']      = _try('OWASP(Python)',      lambda: extract_owasp_benchmark(RAW/'owasp_benchmark_python', language='Python'))
collections['LLMSecEval']         = _try('LLMSecEval',         lambda: extract_llmseceval(RAW/'llmseceval', language='Python'))
collections['SecurityEval']       = _try('SecurityEval',       lambda: extract_security_eval(RAW/'security_eval'))
collections['CAPEC_LLM(Python)']  = _try('CAPEC_LLM(Python)',  lambda: extract_capec_llm(RAW/'capec_llm', language='Python'))

collections = {k: v for k, v in collections.items() if v}
print(f'\nLoaded: {list(collections.keys())}')

## 4. Compute statistics

In [5]:
stats: dict[str, dict] = {}

for name, samples in collections.items():
    vuln    = [s for s in samples if s.label == 1]
    safe    = [s for s in samples if s.label == 0]
    counter = Counter(cwe for s in vuln for cwe in s.cwes)
    branch  = samples[0].branch if samples else ''
    stats[name] = {
        'total': len(samples), 'vulnerable': len(vuln), 'safe': len(safe),
        'unique_cwes': len(counter), 'cwe_counts': counter, 'branch': branch,
    }

print(f'{"Dataset":<26} {"Branch":>6} {"Total":>8} {"Vuln":>8} {"Safe":>8} {"CWEs":>6}')
print('-' * 68)
for n, s in stats.items():
    print(f'{n:<26} {s["branch"]:>6} {s["total"]:>8,} {s["vulnerable"]:>8,} {s["safe"]:>8,} {s["unique_cwes"]:>6,}')

Dataset                    Branch    Total     Vuln     Safe   CWEs
--------------------------------------------------------------------
CVEfixes(Python)             real      511      511        0    105
PatchEval                    real    1,616      808      808     60
CrossVul(Python)             real    1,087      544      543     55
PyVul                        real    3,043    1,569    1,474    138
SVEN(Python)                 real      760      380      380      4
OWASP(Python)               synth    1,230      452      778     14
LLMSecEval                     ai      717      571      146     12
synth-vuln-fixes               ai      252       62      190     23
SecurityEval                   ai      121      121        0     69
CAPEC_LLM(Python)              ai    3,344    3,344        0    510


## 5. Build Excel report

In [6]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

FILL_HEADER     = PatternFill('solid', fgColor='1F4E79')
FILL_LEAF       = PatternFill('solid', fgColor='92D050')
FILL_NONLEAF    = PatternFill('solid', fgColor='FFC000')
FILL_UNKNOWN    = PatternFill('solid', fgColor='BFBFBF')
FILL_DEPRECATED = PatternFill('solid', fgColor='FFB3B3')
FILL_CATEGORY   = PatternFill('solid', fgColor='D2B4DE')
FILL_ROW_ODD    = PatternFill('solid', fgColor='F2F2F2')
FONT_HEADER     = Font(bold=True, color='FFFFFF', name='Calibri', size=11)
FONT_CWE        = Font(bold=True, color='000000', name='Calibri', size=10)
FONT_DATA       = Font(name='Calibri', size=10)
FONT_DATA_B     = Font(bold=True, name='Calibri', size=10)
ALIGN_C = Alignment(horizontal='center', vertical='center', wrap_text=True)
ALIGN_L = Alignment(horizontal='left', vertical='center')
THIN    = Side(style='thin', color='D3D3D3')
BORDER  = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

_CWE_FILL = {'leaf': FILL_LEAF, 'non-leaf': FILL_NONLEAF, 'unknown': FILL_UNKNOWN,
             'deprecated': FILL_DEPRECATED, 'category': FILL_CATEGORY}

FIXED_COLS = ['Dataset', 'Total Samples', 'Vulnerable', 'Safe', 'Unique CWEs']
N_FIXED    = len(FIXED_COLS)

def _write_data_sheet(ws, dataset_names, all_stats):
    global_c = Counter()
    for dn in dataset_names:
        global_c.update(all_stats[dn]['cwe_counts'])
    cwes = [cwe for cwe, _ in global_c.most_common()]

    for ci, col in enumerate(FIXED_COLS, 1):
        cell = ws.cell(1, ci, col)
        cell.fill, cell.font, cell.alignment, cell.border = FILL_HEADER, FONT_HEADER, ALIGN_C, BORDER

    for ci, cwe in enumerate(cwes, N_FIXED + 1):
        num  = _strip(cwe)
        name = nav.get_element_name(num)
        cell = ws.cell(1, ci, f'{cwe}\n{name}' if name != 'Unknown' else cwe)
        cell.fill, cell.font, cell.alignment, cell.border = _CWE_FILL[cwe_label(cwe)], FONT_CWE, ALIGN_C, BORDER

    for ri, dn in enumerate(dataset_names, 2):
        s = all_stats[dn]
        rf = FILL_ROW_ODD if ri % 2 == 1 else PatternFill()
        for ci, val in enumerate([dn, s['total'], s['vulnerable'], s['safe'], s['unique_cwes']], 1):
            cell = ws.cell(ri, ci, val)
            cell.font = FONT_DATA_B if ci == 1 else FONT_DATA
            cell.alignment = ALIGN_L if ci == 1 else ALIGN_C
            cell.fill, cell.border = rf, BORDER
        for ci, cwe in enumerate(cwes, N_FIXED + 1):
            cnt = s['cwe_counts'].get(cwe, 0)
            cell = ws.cell(ri, ci, cnt if cnt > 0 else None)
            cell.font, cell.alignment, cell.fill, cell.border = FONT_DATA, ALIGN_C, rf, BORDER

    ws.column_dimensions['A'].width = 22
    for c in range(2, N_FIXED + 1):
        ws.column_dimensions[get_column_letter(c)].width = 14
    for c in range(N_FIXED + 1, N_FIXED + len(cwes) + 1):
        ws.column_dimensions[get_column_letter(c)].width = 16
    ws.row_dimensions[1].height = 48
    ws.freeze_panes = 'B2'

wb = Workbook()
wb.remove(wb.active)

SHEET_MAP = {'real': 'Py_Real', 'synth': 'Py_Synth', 'ai': 'Py_AI'}
for branch, sheet_name in SHEET_MAP.items():
    datasets = [n for n, s in stats.items() if s['branch'] == branch]
    if datasets:
        ws = wb.create_sheet(sheet_name)
        _write_data_sheet(ws, datasets, stats)

wf = wb.create_sheet('Filters')
filter_cols = ['Dataset', 'Branch', 'Language', 'Source', 'Positives', 'Negatives',
               'CWE Source', 'Filters Applied', 'Safe Type', 'Label Quality', 'Notes']
for ci, col in enumerate(filter_cols, 1):
    cell = wf.cell(1, ci, col)
    cell.fill, cell.font, cell.alignment, cell.border = FILL_HEADER, FONT_HEADER, ALIGN_C, BORDER
for ri, (ds_name, filt) in enumerate(DATASET_FILTERS.items(), 2):
    rf = FILL_ROW_ODD if ri % 2 == 1 else PatternFill()
    row_vals = [ds_name, filt['branch'], filt['language'], filt['source'],
                filt['positives'], filt['negatives'], filt['cwe_source'],
                filt['filters_applied'], filt['safe_type'], filt['label_quality'], filt['notes']]
    for ci, val in enumerate(row_vals, 1):
        cell = wf.cell(ri, ci, val)
        cell.font = FONT_DATA_B if ci == 1 else FONT_DATA
        cell.alignment = ALIGN_L
        cell.fill, cell.border = rf, BORDER
filter_widths = [22, 8, 8, 35, 45, 30, 30, 45, 14, 12, 40]
for ci, w in enumerate(filter_widths, 1):
    wf.column_dimensions[get_column_letter(ci)].width = w
wf.freeze_panes = 'B2'

wl = wb.create_sheet('Legend')
legend_rows = [
    ('Colour', 'Meaning'),
    ('Green  (#92D050)', 'Leaf CWE -- most specific; no children in MITRE tree'),
    ('Amber  (#FFC000)', 'Non-leaf CWE -- parent/intermediate node'),
    ('Gray   (#BFBFBF)', 'Unknown CWE -- ID not in cwec_latest.xml'),
    ('Pink   (#FFB3B3)', 'Deprecated CWE -- name starts with DEPRECATED:'),
    ('Purple (#D2B4DE)', 'Category/View -- MITRE organisational grouping'),
]
fills = [FILL_HEADER, FILL_LEAF, FILL_NONLEAF, FILL_UNKNOWN, FILL_DEPRECATED, FILL_CATEGORY]
for ri, (a, b) in enumerate(legend_rows, 1):
    ca, cb = wl.cell(ri, 1, a), wl.cell(ri, 2, b)
    ca.fill = fills[ri - 1]
    ca.font = FONT_HEADER if ri == 1 else FONT_CWE
    cb.font = FONT_HEADER if ri == 1 else FONT_DATA
    ca.alignment = cb.alignment = ALIGN_L
    ca.border = cb.border = BORDER
wl.column_dimensions['A'].width = 22
wl.column_dimensions['B'].width = 60

wb.save(XLSX_PATH)
print(f'Saved: {XLSX_PATH}')
print(f'Sheets: {wb.sheetnames}')

Saved: C:\Users\franc\OneDrive - Università di Napoli Federico II\Desktop\PhD\FEAST\outputs\stage1_python_stats.xlsx
Sheets: ['Py_Real', 'Py_Synth', 'Py_AI', 'Filters', 'Legend']


## 6. Summary

In [7]:
global_counter: Counter = Counter()
for s in stats.values():
    global_counter.update(s['cwe_counts'])
ALL_CWES = [cwe for cwe, _ in global_counter.most_common()]

by_label = {lbl: [c for c in ALL_CWES if cwe_label(c) == lbl]
            for lbl in ('leaf', 'non-leaf', 'unknown', 'deprecated', 'category')}
for lbl, cwes in by_label.items():
    print(f'  {lbl:<12}: {len(cwes):>4}  {cwes[:5]} ...')

  leaf        :  359  ['CWE-78', 'CWE-601', 'CWE-494', 'CWE-502', 'CWE-294'] ...
  non-leaf    :  185  ['CWE-200', 'CWE-22', 'CWE-79', 'CWE-20', 'CWE-89'] ...
  unknown     :    0  [] ...
  deprecated  :   11  ['CWE-534', 'CWE-247', 'CWE-592', 'CWE-217', 'CWE-533'] ...
  category    :    9  ['CWE-264', 'CWE-255', 'CWE-310', 'CWE-254', 'CWE-19'] ...
